# Vehicle Classification — Improved Pipeline

**What changed vs. the original notebook:**
1. **HOG + color histogram features** instead of raw flattened pixels (~1.8k meaningful features vs 3k–12k pixels). Faster *and* more accurate for Decision Trees.
2. **Proper train/val/test split** — tune hyperparameters on validation, evaluate on test **once** (no data leakage).
3. **Feature caching** — features are extracted once and saved to disk.
4. **RandomForest benchmark** — usually a large accuracy jump over a single tree.

In [1]:
# Run once on a fresh runtime (Kaggle/Colab)
%pip install -q kagglehub split-folders scikit-image

import os, pickle, time
import numpy as np
import cv2
import kagglehub
import splitfolders
from skimage.feature import hog
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Note: you may need to restart the kernel to use updated packages.


/Users/macbookair/Desktop/ML/VE/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Download dataset
path = kagglehub.dataset_download("mohamedmaher5/vehicle-classification")
data_path = os.path.join(path, "Vehicles")
CLASSES = sorted(os.listdir(data_path))
print("Classes:", CLASSES)

 41%|████▏     | 341M/827M [01:36<01:38, 5.17MB/s] 

In [ ]:
# 2. Split: 70% train / 15% validation / 15% test  (no leakage, val used for tuning)
output_path = "/kaggle/working/vehicles_split"   # Colab: /content/vehicles_split

splitfolders.ratio(data_path, output=output_path, seed=42,
                   ratio=(0.7, 0.15, 0.15), group_prefix=None)
print("Split done ->", output_path)

### 3. Feature extraction (the important part)

Raw pixels treat every pixel as an independent feature — a tree has to learn "what does a wheel look like" from single pixels. **HOG** instead describes *edges and shapes*, which is exactly what separates vehicles. We add a small HSV color signature on top.

In [ ]:
IMG_SIZE = 128   # HOG keeps this affordable even at 128x128

def extract_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   feature_vector=True)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv = cv2.resize(hsv, (IMG_SIZE, IMG_SIZE))
    h_hist = cv2.calcHist([hsv], [0], None, [32], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [32], [0, 256]).flatten()
    return np.concatenate([hog_feat,
                           h_hist / max(h_hist.sum(), 1e-9),
                           s_hist / max(s_hist.sum(), 1e-9)]).astype(np.float32)

def load_split(split):
    X, y = [], []
    for label, cat in enumerate(CLASSES):
        folder = os.path.join(output_path, split, cat)
        for name in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, name))
            if img is None:
                continue
            X.append(extract_features(img))
            y.append(label)
    return np.array(X), np.array(y)

CACHE = "/kaggle/working/features.npz"
if os.path.exists(CACHE):
    d = np.load(CACHE)
    X_train, y_train, X_val, y_val, X_test, y_test = (d[k] for k in
        ["X_train","y_train","X_val","y_val","X_test","y_test"])
    print("Loaded cached features.")
else:
    t0 = time.time()
    X_train, y_train = load_split("train")
    X_val,   y_val   = load_split("val")
    X_test,  y_test  = load_split("test")
    np.savez_compressed(CACHE, X_train=X_train, y_train=y_train,
                        X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test)
    print(f"Extracted in {time.time()-t0:.0f}s | train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

### 4. Tune `max_depth` on the **validation** set

Features are now small (~1.8k floats), so each fit is seconds, not minutes. No image re-loading, no leakage.

In [ ]:
best_depth, best_acc = None, 0
for d in range(5, 42, 2):
    t0 = time.time()
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_val, clf.predict(X_val))
    marker = ""
    if acc > best_acc:
        best_depth, best_acc = d, acc
        marker = "  <- best so far"
    print(f"depth={d:2d}  val_acc={acc:.3f}  ({time.time()-t0:.1f}s){marker}")

print(f"\nBest depth: {best_depth} (val acc {best_acc:.3f})")

### 5. RandomForest benchmark + final test evaluation

A single decision tree is a weak learner; an ensemble of them on HOG features is typically far better. We evaluate on test **once**, for both models.

In [ ]:
# Final models trained on train+val (standard practice: more data for the final fit)
X_trval = np.concatenate([X_train, X_val])
y_trval = np.concatenate([y_train, y_val])

dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt.fit(X_trval, y_trval)
dt_test_acc = accuracy_score(y_test, dt.predict(X_test))
print(f"DecisionTree (HOG)  test acc: {dt_test_acc:.3f}")

rf = RandomForestClassifier(n_estimators=300, max_depth=None,
                            n_jobs=-1, random_state=42)
rf.fit(X_trval, y_trval)
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"RandomForest (HOG)  test acc: {rf_test_acc:.3f}")

clf = rf if rf_test_acc >= dt_test_acc else dt
print("\nSelected model:", type(clf).__name__)
print(classification_report(y_test, clf.predict(X_test), target_names=CLASSES))

### 6. Predict a single image + save model

The saved pickle now includes the preprocessing config, so prediction uses **exactly** the same pipeline as training.

In [ ]:
def predict_image(image_path, model):
    img = cv2.imread(image_path)
    if img is None:
        return "Could not read image"
    feat = extract_features(img).reshape(1, -1)
    return CLASSES[int(model.predict(feat)[0])]

print(predict_image("image.jpg", clf))

with open("vehicle_model.pkl", "wb") as f:
    pickle.dump({"model": clf, "classes": CLASSES, "img_size": IMG_SIZE}, f)
print("Model saved -> vehicle_model.pkl")

### 7. Want even higher accuracy? Use a pretrained CNN

If HOG + RandomForest plateaus, transfer learning with MobileNetV2 on this 7-class problem will almost certainly beat any classic ML approach. That's the next step if you need it.